# Copy Files from ABFS Path to Mounted Lakehouse

This notebook shows how to copy a few files from an ABFS (Azure Blob File System) path into the **already-mounted Lakehouse** attached to this notebook, using `notebookutils`.

The default mount point for the attached Lakehouse is `/lakehouse/default`.

**Steps:**
1. Configure the ABFS source path and the files to copy.
2. Copy individual files from the ABFS source into the mounted Lakehouse `Files` area.
3. Verify the copied files.

## 1. Configure source and destination

Set the ABFS source folder. **All files** in that folder will be copied into the destination sub-folder of the mounted Lakehouse.

In [ ]:
import notebookutils

# ABFS source folder (all files in this folder will be copied)
# Format: abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/<folder>
source_abfs_path = "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/source"

# Destination sub-folder inside the mounted Lakehouse Files area
destination_subfolder = "imported"

# Optional: only copy files matching these extensions. Set to None to copy every file.
file_extensions = None  # e.g. [".csv", ".json", ".parquet"]

## 2. Use the default mount point

Since the Lakehouse is already mounted, point at its default local path `/lakehouse/default`. The `Files` area lives under `/lakehouse/default/Files`.

In [ ]:
# Default mount point of the attached Lakehouse
local_mount_path = "/lakehouse/default"
print(f"Using mounted Lakehouse at: {local_mount_path}")

## 3. Copy the files

List every file in the source folder with `notebookutils.fs.ls`, then copy each one into the mounted Lakehouse `Files` area with `notebookutils.fs.cp`. The `file:` scheme points at the local mounted path.

In [ ]:
import os

# Local destination folder inside the mounted Lakehouse Files area
local_dest_folder = os.path.join(local_mount_path, "Files", destination_subfolder)

# Ensure the destination folder exists
os.makedirs(local_dest_folder, exist_ok=True)

# List all entries in the source folder and keep only files (skip sub-folders)
entries = notebookutils.fs.ls(source_abfs_path)
files = [e for e in entries if e.isFile]

# Optional extension filter
if file_extensions:
    allowed = tuple(ext.lower() for ext in file_extensions)
    files = [e for e in files if e.name.lower().endswith(allowed)]

print(f"Found {len(files)} file(s) to copy.")

copied = 0
for e in files:
    src = e.path
    dst = f"file:{os.path.join(local_dest_folder, e.name)}"
    print(f"Copying {e.name} -> {dst}")
    notebookutils.fs.cp(src, dst)
    copied += 1

print(f"Done. Copied {copied} file(s).")

## 4. Verify the copied files

List the destination folder to confirm the files were copied successfully.

In [ ]:
for f in os.listdir(local_dest_folder):
    full_path = os.path.join(local_dest_folder, f)
    size_kb = os.path.getsize(full_path) / 1024
    print(f"{f} ({size_kb:.1f} KB)")